# Hurricane Damage Detector — Exploration

EDA notebook for the Hurricane Harvey satellite imagery dataset.

**Dataset:** Cao & Choe (2018) — [IEEE DataPort](https://ieee-dataport.org/open-access/detecting-damaged-buildings-post-hurricane-satellite-imagery-based-customized) | [Kaggle Mirror](https://www.kaggle.com/datasets/kmader/satellite-images-of-hurricane-damage)

**Contents:**
1. Dataset overview and class distribution
2. Sample images grid (damage vs no_damage)
3. Image statistics (RGB channel means and stds)
4. Data augmentation demo

> **Note:** This notebook is for exploration only. Training and evaluation run via `python -m src.train`.

In [ ]:
import sys
sys.path.append("..")

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from config import (
    TRAIN_DIR, VALIDATION_DIR, TEST_DIR,
    IMG_DIMS, BATCH_SIZE, CLASS_NAMES, SEED,
)
from src.data import download_and_extract_data

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# Download dataset if not present
download_and_extract_data()

## 1. Dataset Overview

In [ ]:
def count_images(base_dir: Path) -> dict:
    """Count images per class in a directory."""
    counts = {}
    for cls in CLASS_NAMES:
        cls_dir = base_dir / cls
        if cls_dir.exists():
            counts[cls] = len(list(cls_dir.glob("*")))
        else:
            counts[cls] = 0
    return counts


splits = {
    "Train": TRAIN_DIR,
    "Validation": VALIDATION_DIR,
    "Test": TEST_DIR,
}

print(f"{'Split':<12} {'no_damage':>10} {'damage':>10} {'Total':>10} {'Balance':>10}")
print("-" * 55)

for name, path in splits.items():
    counts = count_images(path)
    total = sum(counts.values())
    ratio = counts['damage'] / total * 100 if total > 0 else 0
    print(f"{name:<12} {counts['no_damage']:>10} {counts['damage']:>10} {total:>10} {ratio:>9.1f}%")

In [ ]:
# Class distribution bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, path) in zip(axes, splits.items()):
    counts = count_images(path)
    bars = ax.bar(counts.keys(), counts.values(), color=["#2ecc71", "#e74c3c"])
    ax.set_title(f"{name} Set", fontweight="bold")
    ax.set_ylabel("Number of Images")
    for bar, val in zip(bars, counts.values()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 50,
            str(val),
            ha="center",
            fontweight="bold",
        )

plt.suptitle("Class Distribution Across Splits", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Sample Images

In [ ]:
# Load a batch for visualization
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    str(TRAIN_DIR),
    class_names=CLASS_NAMES,
    seed=SEED,
    image_size=IMG_DIMS,
    batch_size=32,
)

# Get one batch
images, labels = next(iter(train_ds))

# Separate by class
no_damage_imgs = images[labels == 0]
damage_imgs = images[labels == 1]

fig, axes = plt.subplots(2, 8, figsize=(20, 5))

for i in range(min(8, len(no_damage_imgs))):
    axes[0, i].imshow(no_damage_imgs[i].numpy().astype("uint8"))
    axes[0, i].axis("off")
    if i == 0:
        axes[0, i].set_title("NO DAMAGE", fontweight="bold", color="green", fontsize=12)

for i in range(min(8, len(damage_imgs))):
    axes[1, i].imshow(damage_imgs[i].numpy().astype("uint8"))
    axes[1, i].axis("off")
    if i == 0:
        axes[1, i].set_title("DAMAGE", fontweight="bold", color="red", fontsize=12)

plt.suptitle("Sample Images from Training Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Image Statistics

Compute per-channel (RGB) mean and standard deviation across the training set.
This helps verify data quality and understand the distribution of pixel values.

In [ ]:
# Compute RGB statistics over a sample of batches
channel_sum = np.zeros(3)
channel_sq_sum = np.zeros(3)
pixel_count = 0

for img_batch, _ in train_ds.take(50):  # Sample 50 batches
    img_np = img_batch.numpy() / 255.0  # Normalize to [0, 1]
    channel_sum += img_np.sum(axis=(0, 1, 2))
    channel_sq_sum += (img_np ** 2).sum(axis=(0, 1, 2))
    pixel_count += img_np.shape[0] * img_np.shape[1] * img_np.shape[2]

channel_mean = channel_sum / pixel_count
channel_std = np.sqrt(channel_sq_sum / pixel_count - channel_mean ** 2)

print("RGB Channel Statistics (normalized to [0, 1]):")
print(f"  Mean: R={channel_mean[0]:.4f}, G={channel_mean[1]:.4f}, B={channel_mean[2]:.4f}")
print(f"  Std:  R={channel_std[0]:.4f}, G={channel_std[1]:.4f}, B={channel_std[2]:.4f}")
print(f"\nFor reference, ImageNet mean: R=0.485, G=0.456, B=0.406")

## 4. Data Augmentation Demo

Visual check that augmentations preserve meaningful features.
Since this is satellite imagery, flips and contrast changes are safe
(no fixed orientation, variable lighting conditions).

In [ ]:
# Pick one sample image
sample_img = images[0]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Original
axes[0, 0].imshow(sample_img.numpy().astype("uint8"))
axes[0, 0].set_title("Original", fontweight="bold")
axes[0, 0].axis("off")

# Augmented variants
augmentations = [
    ("Horizontal Flip", lambda img: tf.image.flip_left_right(img)),
    ("Vertical Flip", lambda img: tf.image.flip_up_down(img)),
    ("Contrast (low)", lambda img: tf.image.adjust_contrast(img, 0.3)),
    ("Contrast (high)", lambda img: tf.image.adjust_contrast(img, 1.5)),
    ("H+V Flip", lambda img: tf.image.flip_up_down(tf.image.flip_left_right(img))),
    ("Random Contrast 1", lambda img: tf.image.random_contrast(img, 0.2, 1.5)),
    ("Random Contrast 2", lambda img: tf.image.random_contrast(img, 0.2, 1.5)),
]

positions = [(0, 1), (0, 2), (0, 3), (1, 0), (1, 1), (1, 2), (1, 3)]

for (title, fn), (r, c) in zip(augmentations, positions):
    aug_img = fn(sample_img)
    aug_img = tf.clip_by_value(aug_img, 0, 255)
    axes[r, c].imshow(aug_img.numpy().astype("uint8"))
    axes[r, c].set_title(title)
    axes[r, c].axis("off")

plt.suptitle("Data Augmentation Demo", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Image Size Verification

Verify that all images have consistent dimensions (expected: 128×128×3).

In [ ]:
from PIL import Image
from collections import Counter

size_counter = Counter()
sample_count = 0

for cls in CLASS_NAMES:
    cls_dir = TRAIN_DIR / cls
    if not cls_dir.exists():
        continue
    for img_path in list(cls_dir.glob("*"))[:500]:  # Sample 500 per class
        try:
            with Image.open(img_path) as pil_img:
                size_counter[pil_img.size] += 1
                sample_count += 1
        except Exception:
            pass

print(f"Checked {sample_count} images from training set:")
for size, count in size_counter.most_common():
    print(f"  {size[0]}x{size[1]}: {count} images ({count/sample_count*100:.1f}%)")